# Семинар 12. Гипотеза о среднем: одна выборка и доверительные интервалы

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-12.ipynb`, например `2MP9-Ivanov-Ivan-seminar-12.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Файлов ноутбук не пишет.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-12-tasks.md` с формулировками и ноутбук `seminar-12-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 12».
- Шпаргалка по функциям: `reference/statistics.md`, раздел «Одна выборка». Сегодня выдаётся этап 3 проекта — проверка гипотез на своём датасете, `project/03-hypotheses.md`, срок — семинар 14; формат записи гипотезы показывает раздел 7.

Что сегодня. На семинаре 11 гипотеза была о форме распределения. Сегодня — первые содержательные вопросы к данным, на которые отвечает одна выборка: равен ли средний стаж водителей, попавших в ДТП, заданному числу; где лежит настоящее среднее и насколько точно оно известно; отличается ли район от области; как проверить не среднее, а медиану; верна ли заданная доля — ДТП с погибшими, ДТП по выходным; поровну ли ДТП по дням недели; что происходит, когда гипотез много. Каждая проверка идёт по одной схеме: вопрос, нулевая и альтернативная гипотезы, тест, p-значение, решение, вывод словами. По той же схеме записываются гипотезы этапа 3.

## 1. Вопрос → гипотеза → тест → вывод

Проверка гипотезы начинается не с теста, а с вопроса. Вопрос: средний стаж водителей, попавших в ДТП, — 15 лет? Число 15 здесь условное. В юридической работе такое число приходит извне — из нормы, отчёта, заявления стороны, — а данные проверяют, согласуется ли оно с ними. Логика проверки та же, что на семинаре 11 для нормальности: меняются только гипотезы и тест.

Подключение — как на семинаре 11, и одна новая строка: функции для долей `proportions_ztest` и `proportion_confint` из модуля `statsmodels.stats.proportion`, они понадобятся в разделе 5. Данные — водители Брянской области с их ДТП, подготовка та же: стаж без кодов 95, 96 и 99, возраст машины — год ДТП минус год выпуска, больше 100 лет — пропуск. Возраст машины сегодня не понадобится, но таблица водителей собирается так же, как в упражнениях.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
drivers = people[people['role'] == 'Водитель'].merge(dtp, left_on='dtp_id', right_index=True)   # водители с их ДТП
drivers['vehicle_age'] = drivers['datetime'].dt.year - drivers['vehicle_year']   # возраст машины в годах
drivers['vehicle_age'] = drivers['vehicle_age'].where(drivers['vehicle_age'] <= 100)   # невозможный возраст в пропуск
experience = drivers['experience_clean'].dropna()                        # стаж без пропусков
print(len(experience), round(experience.mean(), 2), experience.median())

Стаж известен у 14 028 водителей, среднее 15,95 года, медиана 13. Гипотезы:

- нулевая H0: средний стаж водителей, попавших в ДТП, равен 15 годам, μ = 15;
- альтернативная H1: не равен, μ ≠ 15. Альтернатива двусторонняя: интересно отклонение в любую сторону.

μ — среднее генеральной совокупности, всех водителей, которые попадают в такие ДТП; 14 028 водителей — выборка из неё. Среднее выборки 15,95 не равно 15, но выборочное среднее всегда немного отличается от настоящего, и вопрос в том, может ли разница почти в год быть случайной. Отвечает t-тест одной выборки, `stats.ttest_1samp(значения, μ0)`. Его статистика t — разница средних, измеренная в стандартных ошибках:

t = (среднее выборки − μ0) / (s / √n),

где s — стандартное отклонение, n — размер выборки, а s / √n — стандартная ошибка среднего, разброс выборочных средних по центральной предельной теореме. Нормальность стажа отвергнута на семинаре 11, но значений тысячи, и t-тест годится: нормально распределено среднее.

In [ ]:
result = stats.ttest_1samp(experience, 15)                               # t-тест против 15
result

Результат — объект `TtestResult`: поле `statistic` — t, `pvalue` — p-значение, `df` — число степеней свободы, n − 1. Та же t получается по формуле; стандартную ошибку считает `stats.sem`.

In [ ]:
se = stats.sem(experience)                                               # стандартная ошибка среднего
print(round(se, 3), round((experience.mean() - 15) / se, 2))             # t по формуле
'отвергаем' if result.pvalue < 0.05 else 'не отвергаем'                  # решение при 0,05

Стандартная ошибка 0,102 года, t 9,34: среднее выборки отстоит от 15 больше чем на девять стандартных ошибок. p 1,1·10⁻²⁰ меньше 0,05 — нулевую гипотезу отвергаем. Вывод словами: средний стаж водителей, попавших в ДТП в Брянской области, отличается от 15 лет (t-тест одной выборки, n = 14 028, t = 9,34, p < 0,001); выборочное среднее 15,95 — почти на год больше.

Оговорка о практической значимости. Тест отвергает 15 не потому, что разница велика, а потому, что выборка огромна и стандартная ошибка — десятая доля года: такая выборка видит и малую разницу. Значимость значит «разница не случайна», важность — «разница имеет значение для дела», и важность решает не тест, а юрист: для оценки опыта водителей год при среднем стаже около 16 лет, скорее всего, ничего не меняет. Поэтому в выводе рядом с p пишут саму разницу.

Ловушка без ошибки — тот же тест по столбцу таблицы как есть, с пропусками:

In [ ]:
gaps = stats.ttest_1samp(drivers['experience_clean'], 15)                # тест по столбцу с пропусками
print(gaps)
'отвергаем' if gaps.pvalue < 0.05 else 'не отвергаем'

Ни ошибки, ни ответа: t, p и даже число степеней свободы — `nan`, потому что в столбце пропуски, а функции scipy их не пропускают. Хуже того, решение записалось как «не отвергаем»: сравнение `nan < 0.05` даёт `False`. Правило то же, что для тестов нормальности: перед тестом — `dropna()`.

## 2. Доверительный интервал

Тест ответил: не 15. А сколько? Выборочное среднее 15,95 — точечная оценка; настоящее среднее где-то рядом, и доверительный интервал показывает где. 95-процентный доверительный интервал строится так, что в 95 выборках из 100 он накрывает настоящее среднее. Формула:

среднее ± t* · s / √n,

где t* — критическое значение распределения Стьюдента с n − 1 степенями свободы. Считает интервал `stats.t.interval(0.95, n − 1, loc=среднее, scale=стандартная ошибка)`: первый аргумент — уровень доверия, второй — число степеней свободы, `loc` и `scale` — центр и масштаб, как у `stats.norm.pdf` на семинаре 11.

In [ ]:
low, high = stats.t.interval(0.95, len(experience) - 1, loc=experience.mean(), scale=se)   # 95-процентный интервал
print(round(low, 2), round(high, 2))

С 95-процентной уверенностью средний стаж водителей, попавших в ДТП, — от 15,75 до 16,15 года. 15 в интервал не попадает: это та же новость, что дал тест.

При запасе времени — интервал по формуле руками. Критическое значение t* даёт `stats.t.ppf` — квантиль распределения Стьюдента. 95 % посередине оставляют по 2,5 % в каждом хвосте, поэтому нужен квантиль 0,975.

In [ ]:
t_star = stats.t.ppf(0.975, len(experience) - 1)                         # критическое значение t
half = t_star * experience.std() / np.sqrt(len(experience))              # половина ширины интервала
print(round(t_star, 3), round(experience.mean() - half, 2), round(experience.mean() + half, 2))
print(round(stats.t.ppf(0.975, 9), 2))                                   # t* для десяти значений

t* 1,96: при n в тысячи распределение Стьюдента не отличается от нормального. На малых выборках t* больше: для десяти значений, девяти степеней свободы, — 2,26, и интервал шире. Это плата за то, что отклонение оценено по немногим значениям.

Интервал и тест — две записи одного ответа: интервал состоит из всех значений μ0, которые t-тест при уровне 0,05 не отвергает. Проверим значения по обе стороны от границ 15,75 и 16,15:

In [ ]:
for mu in [15.7, 15.8, 16.1, 16.2]:
    p = stats.ttest_1samp(experience, mu).pvalue                         # тест против mu
    print(mu, round(p, 3), 'отвергаем' if p < 0.05 else 'не отвергаем')

15,8 и 16,1 внутри интервала — не отвергаем; 15,7 и 16,2 снаружи — отвергаем. Поэтому в отчёте интервал полезнее одного p: он отвечает и «отличается ли», и «насколько». А «не отвергаем 16,1» не значит, что среднее равно 16,1: не отвергнуты ни 15,8, ни всё, что между ними.

Ширина интервала — 2 · t* · s / √n: чем больше выборка, тем уже интервал, но сужается он не пропорционально n, а как корень из n. Сравним всю область с одним районом — Жуковским:

In [ ]:
district = drivers.loc[drivers['district'] == 'Жуковский район', 'experience_clean'].dropna()   # стаж водителей района
d_low, d_high = stats.t.interval(0.95, len(district) - 1, loc=district.mean(), scale=stats.sem(district))
print(len(district), round(district.mean(), 2), round(d_low, 2), round(d_high, 2))
print(round(high - low, 2), round(d_high - d_low, 2))                    # ширины двух интервалов
print(round((d_high - d_low) / (high - low), 2), round(np.sqrt(len(experience) / len(district)), 2))   # отношение ширин и корень

В Жуковском районе 348 водителей со стажем, среднее 14,89, интервал от 13,64 до 16,14 — шириной 2,50 года против 0,40 у всей области. Интервал района шире в 6,26 раза — почти как корень из отношения размеров выборок, 6,35. Чтобы сузить интервал вдвое, выборку нужно увеличить вчетверо.

## 3. Район как выборка

Вопрос: отличается ли средний стаж водителей Жуковского района от области? Генеральная совокупность — все водители области, её среднее известно: 15,95. Выборка — водители района. Так часть сравнивают с целым без внешних нормативов: район с областью, суд с судами региона. Строго говоря, район — не случайная выборка из области, и тест отвечает на вопрос: могли бы водители района оказаться такими случайно, если бы район ничем не отличался от области?

Но район сам входит в область, и сколько он в ней весит, зависит от его доли среди водителей области:

In [ ]:
shares = drivers.loc[drivers['experience_clean'].notna(), 'district'].value_counts(normalize=True)   # доли районов
print(round(shares['Жуковский район'] * 100, 1), round(shares['Брянский район'] * 100, 1))

Водители Жуковского района — 2,5 % водителей области со стажем, Брянского — 13,9 %. Считать среднее области известным числом — приближение: оно посчитано по тем же водителям, что и среднее района, и само неточно — в разделе 2 у него был интервал. Для малого района приближение годится: его водители почти не сдвигают областное среднее, а интервал областного среднего, шириной 0,40 года, в шесть раз уже интервала района. Для большого района оно не годится: Брянский район заметно тянет областное среднее к своему, и сравнение выходит отчасти сравнением района с самим собой. Большой район сравнивают не с областью, а с остальной областью — тестом для двух выборок, это семинар 13.

Гипотезы: нулевая — среднее района равно областному, μ = 15,95; альтернативная — не равно. Среднее области передаётся тесту числом, как раньше 15. У параметра `alternative` три значения: `'two-sided'` — не равно, по умолчанию; `'greater'` — больше; `'less'` — меньше.

In [ ]:
two_sided = stats.ttest_1samp(district, experience.mean())               # двусторонняя альтернатива
less = stats.ttest_1samp(district, experience.mean(), alternative='less')   # среднее района меньше
print(round(two_sided.statistic, 2), round(two_sided.pvalue, 3), round(less.pvalue, 3))

t −1,67, двусторонний p 0,096 — больше 0,05: не отвергаем. Односторонний, с альтернативой «среднее района меньше областного», — p 0,048: отвергаем. Односторонний p вдвое меньше: тест смотрит только в один хвост распределения t, и то же отклонение в нём вдвое «дороже».

Правило: альтернативу выбирают до того, как посмотрели на данные, — из самого вопроса. Если вопрос «у водителей района стаж меньше?» поставлен заранее — из жалобы, отчёта, прошлогоднего анализа, — односторонний тест честен. Если же сначала увидели среднее 14,89, а потом выбрали «меньше», уровень значимости на деле не 5 %, а 10 %: при среднем выше областного выбрали бы «больше», и ложное «отвергаем» набиралось бы из обоих хвостов.

Ловушка без ошибки — альтернатива по данным. Та же проверка с альтернативой в другую сторону:

In [ ]:
greater = stats.ttest_1samp(district, experience.mean(), alternative='greater')   # среднее района больше
print(round(greater.pvalue, 3))

p 0,952: в сторону «больше» данные не дают ничего, ведь среднее района ниже областного. Кто выбирает направление после взгляда на данные, всегда получает меньший из двух односторонних p — половину двустороннего. Поэтому по умолчанию тест двусторонний, а односторонний — только с направлением, записанным до анализа.

И ещё: «не отвергаем» при двустороннем p 0,096 не значит «район такой же, как область». Интервал района — от 13,64 до 16,14: средний стаж в районе может быть таким же, как в области, а может быть меньше на два с лишним года. Данных не хватило, чтобы решить, — и это всё, что значит «не отвергаем».

## 4. Медиана: знаковый тест

t-тест сравнивает среднее, а у стажа правый хвост: среднее 15,95, медиана 13 — среднее тянут водители с большим стажем. Если вопрос о типичном водителе, а не о среднем, проверяют медиану: половина водителей, попавших в ДТП, имеет стаж больше m0 лет? Тест медианы — знаковый. Если медиана равна m0, каждый водитель с равной вероятностью оказывается выше или ниже m0, как монета ложится орлом или решкой. Значения, равные m0, отбрасывают, а число значений выше m0 среди остальных сравнивают с половиной биномиальным тестом `stats.binomtest(k, n)`: k — сколько значений выше m0, n — сколько значений не равно m0. Результат — объект `BinomTestResult` с полями `statistic` — доля k / n — и `pvalue`. Тест смотрит только на знак отклонения, не на величину, поэтому длинный хвост и выбросы на него не влияют.

In [ ]:
for m0 in [15, 13]:
    above, n_signs = int((experience > m0).sum()), int((experience != m0).sum())   # выше m0; всего без равных
    print(m0, above, n_signs - above, stats.binomtest(above, n_signs).pvalue)   # знаковый тест

Против 15: выше — 6 034 водителя, ниже — 7 572. Будь медиана 15, они делились бы почти поровну, а перевес в полторы тысячи на 13 606 водителях случайным не бывает: p 9,9·10⁻⁴⁰ — отвергаем. Медиана меньше 15, хотя среднее больше: t-тест отверг 15 сверху, знаковый тест — снизу. Тесты отвечают на разные вопросы: t-тест — о среднем, знаковый — о медиане; выбирают тест по вопросу и до данных, а не по тому, у какого p меньше. Против 13, медианы самой выборки, выше 6 840 водителей, ниже 6 770, p 0,55 — не отвергаем. Вывод словами: медиана стажа водителей, попавших в ДТП в Брянской области, меньше 15 лет (знаковый тест, n = 13 606, p < 0,001).

Тест, который часто называют тестом медианы, — знаковый ранговый тест Вилкоксона, `stats.wilcoxon(x - m0)`, — проверяет другое. Он нумерует отклонения от m0 по модулю, от меньшего к большему, — это ранги — и сравнивает суммы рангов положительных и отрицательных отклонений; нулевая гипотеза — отклонения симметричны относительно нуля. У симметричного распределения центр симметрии и есть медиана, и там тест Вилкоксона проверяет её точнее знакового: он видит не только знак отклонения, но и его величину. У скошенного распределения он отвечает на вопрос о симметрии, а не о медиане. Ловушка без ошибки — тест Вилкоксона против медианы самой выборки:

In [ ]:
stats.wilcoxon(experience - 13).pvalue                                   # Вилкоксон против медианы

p 1,3·10⁻⁷⁵ — «отвергаем», хотя 13 — медиана выборки и знаковый тест её не отверг. Выше и ниже 13 водителей почти поровну, но отклонения вверх, в длинный хвост, по модулю больше, и их ранги перевешивают. Тест не ошибся — стаж действительно не симметричен относительно 13; ошибкой был бы вывод «медиана не 13». Для медианы скошенного столбца — знаковый тест, тест Вилкоксона — для данных, близких к симметричным.

## 5. Доля

Многие юридические вопросы — о долях: доля ДТП с погибшими, доля обжалованных решений, доля удовлетворённых исков. Доля — среднее логического столбца, и для неё есть свой тест — z-тест для доли, `proportions_ztest(count, nobs, value, prop_var=value)` из statsmodels. Аргументы по порядку: `count` — число «успехов», здесь ДТП с погибшими; `nobs` — число наблюдений, все ДТП; `value` — проверяемая доля p0. Статистика

z = (p̂ − p0) / √(p0 · (1 − p0) / n),

где p̂ — доля в выборке. Стандартная ошибка в знаменателе считается по проверяемой доле p0 — это и задаёт `prop_var=value`, и так тест согласован с интервалом Вилсона ниже, а без `prop_var` statsmodels считает её по доле выборки p̂: это другой тест, тест Вальда. Результат — кортеж из двух чисел (z, p) без имён полей, как у `lilliefors`.

Вопрос: доля ДТП с погибшими в Брянской области — 15 %? Нулевая гипотеза: p = 0,15; альтернативная: p ≠ 0,15. ДТП с погибшими — те, где `dead_count` больше нуля; считаются по таблице ДТП, где одно ДТП — одна строка.

In [ ]:
dead = dtp['dead_count'] > 0                                             # ДТП с погибшими
count, nobs = dead.sum(), len(dtp)
z, p = proportions_ztest(count, nobs, 0.15, prop_var=0.15)               # z-тест для доли
print(count, nobs, round(count / nobs, 4), round(z, 2), p)

1 359 ДТП с погибшими из 10 114, доля 0,1344; z −4,40, p 1,1·10⁻⁵ — отвергаем. Вывод словами: доля ДТП с погибшими в Брянской области меньше 15 % (z-тест для доли, n = 10 114, z = −4,40, p < 0,001).

Интервал для доли — `proportion_confint(count, nobs, method='wilson')`. По умолчанию функция строит интервал по нормальному приближению: доля ± 1,96 стандартной ошибки. На малых выборках и при долях у нуля он ошибается: граница упирается в ноль, а при нуле успехов интервал схлопывается в точку. Интервал Вилсона таких провалов не даёт, и в курсе используется он.

In [ ]:
low_share, high_share = proportion_confint(count, nobs, method='wilson')   # интервал Вилсона
print(round(low_share, 4), round(high_share, 4))
print(round(proportions_ztest(count, nobs, 0.14, prop_var=0.14)[1], 3))   # тест против 0,14

Интервал от 0,1279 до 0,1412: 0,15 вне его — и тест её отверг; 0,14 внутри — и тест против 0,14 даёт p 0,103, не отвергаем. Это не совпадение: интервал Вилсона состоит ровно из тех долей, которые z-тест с `prop_var` при уровне 0,05 не отвергает. Как читать интервал юристу: с 95-процентной уверенностью ДТП с погибшими составляют от 12,8 до 14,1 % ДТП с пострадавшими в Брянской области. Это оценка для того же потока ДТП, а не вероятность гибели в отдельном ДТП и не доля для другой области: на других дорогах и при другом учёте доля другая.

Ловушка без ошибки — тест без `prop_var`, тест Вальда. На тысячах ДТП он почти всегда приходит к тому же решению, что и тест с `prop_var`, а на малой выборке решения расходятся. Пусть в районе 40 ДТП, из них 10 с погибшими:

In [ ]:
print([round(b, 3) for b in proportion_confint(10, 40, method='wilson')])   # интервал Вилсона
for p0 in [0.40, 0.14]:
    wald = proportions_ztest(10, 40, p0)[1]                              # ошибка по доле выборки
    score = proportions_ztest(10, 40, p0, prop_var=p0)[1]                # ошибка по p0
    print(p0, round(wald, 3), round(score, 3))

Интервал Вилсона — от 0,142 до 0,402. 0,40 внутри него, и тест с `prop_var` её не отвергает, p 0,053, а тест Вальда отвергает, p 0,028. 0,14 снаружи — тест с `prop_var` её отвергает, p 0,045, а тест Вальда нет, p 0,108. Тест Вальда противоречит интервалу в обе стороны: его стандартная ошибка посчитана по доле выборки, 10 / 40, а не по проверяемой доле. Поэтому рядом с интервалом Вилсона — тест с `prop_var=value`.

Ловушка без ошибки — аргументы `proportions_ztest` в другом порядке. Кажется естественным написать сначала «всего», потом «сколько»:

In [ ]:
proportions_ztest(nobs, count, 0.15, prop_var=0.15)                      # аргументы перепутаны

statsmodels не проверяет, что успехов не больше, чем наблюдений: «доля» 10 114 / 1 359 больше единицы, z 752,9 и p 0 — «отвергаем» без ошибки и без предупреждения. Порядок аргументов: сначала сколько успехов, потом сколько всего, потом проверяемая доля. Надёжнее писать аргументы по имени: `proportions_ztest(count=count, nobs=nobs, value=0.15, prop_var=0.15)`.

Второй вопрос о доле — ДТП по выходным. Если бы день недели не влиял на ДТП, на субботу и воскресенье приходилось бы 2/7 всех ДТП. Проверяемая доля здесь — не норма закона, а арифметика календаря. Номер дня недели — `dt.dayofweek`: 0 — понедельник, 5 — суббота, 6 — воскресенье.

In [ ]:
weekend = dtp['datetime'].dt.dayofweek >= 5                              # суббота и воскресенье
z_w, p_w = proportions_ztest(weekend.sum(), len(dtp), 2 / 7, prop_var=2 / 7)   # против двух седьмых
print(weekend.sum(), round(weekend.mean(), 4), round(2 / 7, 4), round(z_w, 2), round(p_w, 3))

3 002 ДТП по выходным, доля 0,2968 против 0,2857; z 2,47, p 0,013 — отвергаем: на выходные приходится больше ДТП, чем две седьмых. Разница — около процентного пункта. Какие именно дни перегружены, этот тест не говорит: он видит только сумму двух дней. Ответ — в разделе 6.

При запасе времени — интервал по нормальному приближению рядом с интервалом Вилсона: на всей области и на самом маленьком районе, Рогнединском.

In [ ]:
small = dtp[dtp['district'] == 'Рогнединский район']                     # самый маленький район
for k, n in [(count, nobs), ((small['dead_count'] > 0).sum(), len(small))]:
    normal = proportion_confint(k, n)                                    # нормальное приближение
    wilson = proportion_confint(k, n, method='wilson')                   # интервал Вилсона
    print(n, k, [round(b, 3) for b in normal], [round(b, 3) for b in wilson])

На 10 114 ДТП интервалы совпадают до третьего знака. В Рогнединском районе 28 ДТП, из них 6 с погибшими: нормальный интервал от 0,062 до 0,366 симметричен вокруг доли, а интервал Вилсона, от 0,102 до 0,395, сдвинут от нуля — доля не может быть отрицательной, и неопределённость в сторону нуля меньше. На малых выборках методы расходятся на несколько процентных пунктов.

## 6. Согласие с распределением

Доля выходных — один вопрос о днях недели. Общий вопрос: поровну ли ДТП по семи дням? Нулевая гипотеза: на каждый день приходится 1/7 ДТП, распределение равномерное; альтернативная: не равномерное. Тест — хи-квадрат согласия, `stats.chisquare(наблюдаемые, ожидаемые)`. Он сравнивает наблюдаемые счётчики с ожидаемыми:

χ² = Σ (наблюдаемое − ожидаемое)² / ожидаемое,

сумма по всем дням. Счётчики по дням недели — `value_counts` по номеру дня, упорядоченные `sort_index`.

In [ ]:
counts = dtp['datetime'].dt.dayofweek.value_counts().sort_index()        # ДТП по дням недели
counts.tolist()

От 1 361 ДТП в среду до 1 607 в пятницу. При равномерном распределении на каждый день приходится 1/7 ДТП. Ошибка в следующей ячейке намеренная.

In [ ]:
stats.chisquare(counts, [1 / 7] * 7)                                     # ожидаемые долями

`ValueError: For each axis slice, the sum of the observed frequencies must agree with the sum of the expected frequencies…`: сумма наблюдаемых — 10 114 ДТП, сумма ожидаемых — единица, и scipy отказывается их сравнивать. Ожидаемые задаются в тех же единицах, что наблюдаемые: не доли, а числа ДТП — доля, умноженная на число всех ДТП, `counts.sum() / 7` на каждый день.

In [ ]:
expected = pd.Series(counts.sum() / 7, index=counts.index)               # ожидаемые ДТП в день
chi = stats.chisquare(counts, expected)                                  # хи-квадрат согласия
print(round(expected.iloc[0], 1), round(chi.statistic, 1), chi.pvalue)

Ожидаемо по 1 444,9 ДТП в день; χ² 43,1, p 1,1·10⁻⁷ — отвергаем. Вывод словами: ДТП в Брянской области распределены по дням недели не поровну (хи-квадрат согласия, n = 10 114, χ² = 43,1, p < 0,001). Степеней свободы у теста 7 − 1 = 6: при заданной сумме последний день определён остальными. Без второго аргумента `chisquare` и так сравнивает с равномерным распределением, но явные ожидаемые понятнее и годятся для любых долей.

Какие дни дают отклонение? Статистика — сумма слагаемых по дням; самое большое слагаемое — самый неравномерный день. Рядом — столбцы ДТП по дням и линия ожидаемого.

In [ ]:
contribution = (counts - expected) ** 2 / expected                       # вклад каждого дня
days = ['пн', 'вт', 'ср', 'чт', 'пт', 'сб', 'вс']
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(days, counts)                                                     # наблюдаемые ДТП
ax.axhline(expected.iloc[0], color='red')                                # ожидаемые при равенстве
ax.set_title('ДТП по дням недели и равномерное распределение')
ax.set_xlabel('День недели')
ax.set_ylabel('Число ДТП')
contribution.round(1).tolist()

Пятница даёт 18,2 из 43,1, суббота — 11,4: эти два дня перегружены, понедельник — четверг чуть ниже линии, воскресенье почти на ней. Отсюда и доля выходных из раздела 5: выходные — это перегруженная суббота плюс обычное воскресенье. Хи-квадрат отвечает «не поровну», вклад и график — «где».

## 7. Много проверок сразу и этап 3

Уровень значимости 0,05 значит: если нулевая гипотеза верна, тест ошибётся в одном случае из двадцати. Для одной проверки это приемлемо. А если проверок много? Вопрос: в каких районах доля ДТП с погибшими отличается от доли по всем районам области? Для каждого района с 50 и более ДТП — z-тест доли против доли по районам; p собираются списком по парам «ДТП с погибшими, все ДТП».

Строку «Брянская область» в проверку не берём. Это не район, а 3 615 ДТП без указания района, и по координатам почти все они — в самом Брянске, который в таблице отдельным значением не выделен; подробнее — в описании данных `datasets/legal/README.md`. Город с районами не сравнивают: движение в нём другое. Поэтому эта строка не входит ни в список районов, ни в долю, с которой они сравниваются.

In [ ]:
city = dtp['district'] == 'Брянская область'                             # ДТП без района: Брянск
print(round(dead[city].mean(), 3), round(dead[~city].mean(), 3))         # доли с погибшими
share_districts = dead[~city].mean()                                     # доля по районам
by_district = dtp[~city].assign(dead=dead).groupby('district')['dead'].agg(['sum', 'count'])   # ДТП с погибшими по районам
by_district = by_district[by_district['count'] >= 50]                    # районы от 50 ДТП
by_district['p'] = [proportions_ztest(k, n, share_districts, prop_var=share_districts)[1] for k, n in zip(by_district['sum'], by_district['count'])]
print(len(by_district), (by_district['p'] < 0.05).sum(), round(0.05 / len(by_district), 4), (by_district['p'] < 0.05 / len(by_district)).sum())
by_district['доля'] = (by_district['sum'] / by_district['count']).round(3)   # доля ДТП с погибшими
by_district[by_district['p'] < 0.05].sort_values('p')

В Брянске доля ДТП с погибшими 0,052, в районах — 0,180, в три с лишним раза больше. Это черта городского движения: в городе скорости ниже и помощь ближе, чем на трассах районов. С городом общая доля опускается до 0,1344 из раздела 5, и районы сравнивались бы отчасти с городом, а не друг с другом.

26 районов, из них у 6 p меньше 0,05. Но если бы ни один район не отличался от остальных, каждая проверка всё равно ошибалась бы с вероятностью 5 %, и на 26 проверок пришёлся бы в среднем больше чем один случайно «значимый» район. Чем больше проверок, тем вернее среди них найдётся «значимая» — по чистой случайности.

Поправка Бонферрони: при m проверках каждую сравнивают не с 0,05, а с 0,05 / m, здесь 0,0019. Тогда вероятность хотя бы одной ложной «значимости» на все проверки вместе — не больше 5 %. После поправки значим один район — Климовский: доля ДТП с погибшими 0,269 против 0,180 по районам. Мглинский, Унеченский, Клинцовский, Дятьковский и Почепский районы значимы только без поправки: в Клинцовском и Дятьковском доля 0,140 — ниже районной, в трёх других выше. Цена поправки — осторожность: районы с p между 0,0019 и 0,05, может быть, и правда отличаются, но одних этих данных для такого утверждения мало.

Подгонка p-значения, p-hacking, — когда проверок много, а в отчёт попадают только «значимые»: перебирают районы, годы, виды ДТП, выбирают альтернативу после взгляда на данные, убирают «выбросы», пока p не станет меньше 0,05. Каждый шаг по отдельности выглядит невинно, а вместе они превращают 5 % ошибки в фикцию. Защита: гипотезы и альтернативы записывают до анализа, в отчёт идут все проверенные гипотезы, а не только значимые, а при многих проверках — поправка.

### Этап 3 проекта

Сегодня выдаётся этап 3 проекта — [project/03-hypotheses.md](../project/03-hypotheses.md), срок — семинар 14. На своём датасете нужно проверить не меньше четырёх гипотез разных видов: нормальность числового столбца (семинар 11); среднее или доля одной выборки против известного значения (сегодня); сравнение двух групп и связь двух категориальных столбцов (семинар 13). Если гипотез больше пяти — поправка на множественные проверки. Каждая гипотеза записывается в одном формате:

| Пункт | Что писать |
|---|---|
| вопрос | вопрос словами, как его задал бы юрист |
| H0 и H1 | нулевая и альтернативная гипотезы; направление альтернативы — до данных |
| тест и почему | какой тест и почему он годится: вид данных, размер выборки, нормальность |
| n | размер выборки после удаления пропусков |
| статистика и p | число теста и p-значение; p меньше 0,001 — «p < 0,001» |
| интервал или размер эффекта | доверительный интервал или разница, которую нашёл тест |
| вывод | «отвергаем» или «не отвергаем» при 0,05 и вывод словами с оговоркой о практической значимости |

Образец — первая гипотеза занятия, записанная по формату:

- Вопрос: средний стаж водителей, попавших в ДТП, — 15 лет?
- H0 и H1: H0 — μ = 15; H1 — μ ≠ 15, альтернатива двусторонняя.
- Тест и почему: t-тест одной выборки — вопрос о среднем; нормальность стажа отвергнута, но n = 14 028, и по центральной предельной теореме среднее распределено почти нормально.
- n: 14 028.
- Статистика и p: t = 9,34, p < 0,001.
- Интервал: 95-процентный доверительный интервал среднего — от 15,75 до 16,15 года.
- Вывод: отвергаем; средний стаж водителей, попавших в ДТП, отличается от 15 лет и составляет около 16 лет. Разница меньше года значима из-за большой выборки; практического значения для оценки опыта водителей она, скорее всего, не имеет.

В отчёт рядом с подробными записями идёт сводная таблица гипотез — строка на гипотезу. Её собирают из списка словарей, а решение добавляют столбцом; в упражнениях это функция части В.

In [ ]:
report = pd.DataFrame([
    {'вопрос': 'средний стаж 15 лет?', 'тест': 't-тест', 'n': len(experience), 'статистика': result.statistic, 'p': result.pvalue},
    {'вопрос': 'стаж в Жуковском районе как в области?', 'тест': 't-тест', 'n': len(district), 'статистика': two_sided.statistic, 'p': two_sided.pvalue},
    {'вопрос': 'доля ДТП с погибшими 0,15?', 'тест': 'z-тест доли', 'n': nobs, 'статистика': z, 'p': p},
    {'вопрос': 'ДТП по дням недели поровну?', 'тест': 'хи-квадрат', 'n': counts.sum(), 'статистика': chi.statistic, 'p': chi.pvalue},
], index=range(1, 5))
report['решение'] = np.where(report['p'] < 0.05, 'отвергаем', 'не отвергаем')   # решение при 0,05
report.round(3)

Четыре строки, три «отвергаем» и одно «не отвергаем»; p, выведенные нулём, в отчёте записываются как «p < 0,001». Сводная таблица не заменяет записи по формату: в ней нет интервалов и выводов словами, она — оглавление раздела гипотез.

## Итоги

Проверка гипотезы идёт по схеме: вопрос словами, нулевая и альтернативная гипотезы, тест, p, решение «отвергаем» или «не отвергаем» при 0,05, вывод словами с оговоркой о практической значимости. t-тест одной выборки `stats.ttest_1samp(x, μ0)` сравнивает среднее с числом; на больших выборках он значим и при разнице, не имеющей значения для дела. Пропуски дают `nan` и молчаливое «не отвергаем» — сначала `dropna()`. Доверительный интервал `stats.t.interval(0.95, n − 1, loc=среднее, scale=stats.sem(x))` — среднее ± t* · s / √n; он состоит из значений, которые тест не отвергает, и сужается как корень из n. Район против области: генеральная совокупность — область, выборка — район; среднее области как известное число — приближение, годное для малого района, а большой район сравнивают с остальной областью тестом для двух выборок; односторонняя альтернатива `alternative='greater'` или `'less'` вдвое уменьшает p, но выбирается только до данных; «не отвергаем» не значит «равно». Медиана — знаковый тест `stats.binomtest(k, n)`: k — сколько значений выше m0, n — сколько не равно m0; величина отклонений ему не важна, и хвост на него не влияет. Знаковый ранговый тест Вилкоксона `stats.wilcoxon(x - m0)` проверяет симметрию отклонений и годится как тест центра только для симметричных данных: у скошенного стажа он отвергает даже медиану самой выборки. Доля: z-тест `proportions_ztest(count, nobs, value, prop_var=value)` — сначала успехи, потом все наблюдения; стандартная ошибка по проверяемой доле согласует тест с интервалом Вилсона `proportion_confint(count, nobs, method='wilson')`, а тест Вальда, без `prop_var`, на малых выборках интервалу противоречит; переставленные аргументы дают бессмысленную z без ошибки. Согласие с распределением: `stats.chisquare(наблюдаемые, ожидаемые)` — ожидаемые в тех же единицах, что наблюдаемые, иначе `ValueError`; вклад дней показывает, где отклонение. Много проверок — много случайных «значимостей»: поправка Бонферрони делит 0,05 на число проверок; гипотезы записывают до анализа и в отчёт пишут все. Строка «Брянская область» — это город Брянск, и с районами его не сравнивают.

Переходите к упражнениям: откройте `exercises/seminar-12-tasks.ipynb`, формулировки в `exercises/seminar-12-tasks.md`; данные там по Московской области. Функции части Б — инструменты гипотез этапа 3: t-тест одной выборки, интервал для среднего, решение словами, знаковый тест для медианы, z-тест и интервал Вилсона для доли, хи-квадрат согласия, поправка Бонферрони; в части В из них собираются таблица гипотез для отчёта, проверки по районам с поправкой и расчёт объёма выборки. Необязательная задача — первые две гипотезы этапа 3 на своём датасете по формату записи. Этап 3 сдаётся на семинаре 14.